In [1]:
from youtube_comment_downloader import *
import pandas as pd
import time
import praw
from google_play_scraper import reviews, Sort
import requests
from bs4 import BeautifulSoup
import re


# source_labels = {
#             '1': "youtube",
#             '2': "reddit",
#             '3': "googleplay",
#             '4': "steam",
#             '5': "amazon"
#         }
# ----------- EXTRACTION FUNCTIONS -----------

def extract_youtube(url, extractor):
    comments = extractor.get_comments_from_url(url)
    temp = []
    for c in comments:
        temp.append({
            'url': url,
            'author': c.get("author"),
            'Text': c.get("text"),
            'date': c.get("time"),
            'source': "youtube"
        })
    return temp


import requests

import time
import requests

def safe_request(url, headers, retries=3):
    for i in range(retries):
        try:
            response = requests.get(url, headers=headers, timeout=10)
            return response
        except requests.exceptions.RequestException as e:
            print(f"Retry {i+1} failed: {e}")
            time.sleep(2)
    return None

def extract_reddit(url):
    original_url = url
    url = url.rstrip("/") + ".json"

    headers = {
        "User-Agent": "MyRedditApp/1.0 (by u/yourusername)"
    }

    response = safe_request(url, headers)

    if response is None:
        print("❌ Failed after retries")
        return []

    if response.status_code != 200:
        print(f"Status error: {response.status_code}")
        return []

    try:
        data_json = response.json()
    except:
        print("⚠️ Not JSON (blocked)")
        return []

    temp = []

    try:
        comments = data_json[1]['data']['children']
        for c in comments:
            body = c['data'].get('body')
            author = c['data'].get('author')
            created = c['data'].get('created_utc')

            if body:
                temp.append({
                    'url': original_url,
                    'author': author,
                    'Text': body,
                    'date': created,
                    'source': "reddit"
                })
    except:
        raise Exception("Unexpected JSON structure")

    time.sleep(1)

    return temp


def extract_google_play(app_id):
    result, _ = reviews(app_id, count=2000, sort=Sort.NEWEST)

    temp = []
    for r in result:
        temp.append({
            'url': f"googleplay_{app_id}",
            'author': r.get("userName"),
            'Text': r.get("content"),
            'date': r.get("at"),
            'source': "googleplay"
        })
    return temp


def extract_steam(app_id, total_reviews=200):
    url = f"https://store.steampowered.com/appreviews/{app_id}"
    
    params = {
        "json": 1,
        "language": "english",
        "num_per_page": 100
    }

    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    all_reviews = []
    cursor = "*"

    while len(all_reviews) < total_reviews:
        params["cursor"] = cursor
        
        res = requests.get(url, params=params, headers=headers).json()

        for r in res.get("reviews", []):
            all_reviews.append({
                'url': f"steam_{app_id}",
                'author': r['author']['steamid'],
                'Text': r['review'],
                'date': r['timestamp_created'],
                'source': "steam"
            })

        cursor = res.get("cursor")

        # stop if no more data
        if not cursor:
            break

    return all_reviews[:total_reviews]


def extract_amazon(asin):
    url = f"https://www.amazon.com/product-reviews/{asin}"
    headers = {"User-Agent": "Mozilla/5.0"}

    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, 'html.parser')

    temp = []
    reviews_html = soup.find_all("div", {"data-hook": "review"})

    for item in reviews_html:

        author_elem = item.find("span", class_="a-profile-name")
        text_elem = item.find("span", {"data-hook": "review-body"})
        date_elem = item.find("span", {"data-hook": "review-date"})

        temp.append({
            'url': f"amazon_{asin}",
            'author': author_elem.get_text() if author_elem else "Anon",
            'Text': text_elem.get_text(strip=True) if text_elem else "",
            'date': date_elem.get_text() if date_elem else None,
            'source': "amazon"
        })

    return temp


# ----------- USER INPUT -----------

semidata = {}
url_source_map = {}
targets = []

while True:
    category_name = input("\nEnter category (game/product) (-1 to exit): ")
    if category_name.strip() == '-1':
        break

    while True:
        print("\nChoose source:")
        print("1. YouTube | 2. Reddit | 3. Google Play | 4. Steam | 5. Amazon")

        source_choice = input("Enter choice (1-5) or -1 to change category: ")
        if source_choice.strip() == '-1':
            break

        source_labels = {
            '1': "youtube",
            '2': "reddit",
            '3': "googleplay",
            '4': "steam",
            '5': "amazon"
        }

        if source_choice not in source_labels:
            print("Invalid choice.")
            continue

        prompt = "Enter URL: " if source_choice in ['1', '2'] else "Enter AppID/ASIN: "
        target = input(prompt)

        if target.strip() == '-1':
            break

        source = source_labels[source_choice]

        # normalize keys for mapping
        if source == "googleplay":
            key = f"googleplay_{target}"
        elif source == "steam":
            key = f"steam_{target}"
        elif source == "amazon":
            key = f"amazon_{target}"
        else:
            key = target

        semidata[key] = category_name
        url_source_map[key] = source
        targets.append(target)


# ----------- EXTRACTION -----------

extractor = YoutubeCommentDownloader()
data = []

print("\nStarting extraction...")

for key in url_source_map:
    source = url_source_map[key]

    try:
        if source == "youtube":
            data.extend(extract_youtube(key, extractor))

        elif source == "reddit":
            data.extend(extract_reddit(key))

        elif source == "googleplay":
            app_id = key.split("_", 1)[1]
            data.extend(extract_google_play(app_id))

        elif source == "steam":
            app_id = key.split("_", 1)[1]
            data.extend(extract_steam(app_id))

        elif source == "amazon":
            asin = key.split("_", 1)[1]
            data.extend(extract_amazon(asin))

        time.sleep(1.5)

    except Exception as e:
        print(f"Error processing {key}: {e}")

# ----------- FINAL DATAFRAME -----------
from datetime import datetime, timedelta
scraped_date = datetime.now()

def convert_relative_time(value):
    if pd.isna(value):
        return None
    # ----------------------------
    # 🔥 FIXED TIMESTAMP HANDLING
    # ----------------------------
    try:
        # direct int/float
        if isinstance(value, (int, float)):
            return pd.to_datetime(int(value), unit='s')

        value_str = str(value).strip()

        # handles "1775614197" AND "1775614197.0"
        if re.match(r'^\d+(\.\d+)?$', value_str):
            return pd.to_datetime(int(float(value_str)), unit='s')

    except:
        pass

    # ----------------------------
    # CLEAN TEXT
    # ----------------------------
    value = str(value).lower()
    value = (
        value.replace("\xa0", " ")
             .replace("(edited)", "")
             .replace("edited", "")
             .replace("ago", "")
             .replace("•", "")
             .strip()
    )

    if value in ["", "just now"]:
        return scraped_date

    if "yesterday" in value:
        return scraped_date - timedelta(days=1)

    match = re.search(r"(\d+)", value)
    num = int(match.group()) if match else None

    # ----------------------------
    # RELATIVE TIME HANDLING
    # ----------------------------
    if num is not None:
        if any(x in value for x in ["sec", "second"]):
            return scraped_date - timedelta(seconds=num)

        if any(x in value for x in ["min", "mins", "minute"]):
            return scraped_date - timedelta(minutes=num)

        if any(x in value for x in ["hr", "hrs", "hour"]):
            return scraped_date - timedelta(hours=num)

        if "day" in value:
            return scraped_date - timedelta(days=num)

        if "week" in value:
            return scraped_date - timedelta(weeks=num)

        if any(x in value for x in ["mo", "month"]):
            return pd.to_datetime(scraped_date) - pd.DateOffset(months=num)

        if any(x in value for x in ["yr", "year"]):
            return scraped_date - pd.DateOffset(years=num)

    # ----------------------------
    # FINAL FALLBACK
    # ----------------------------
    try:
        return pd.to_datetime(value, errors='coerce')
    except:
        return None

if data:
    df = pd.DataFrame(data)

    # category mapping
    df['category'] = df['url'].map(semidata)

    # date normalization
    df['parsed_date'] = df['date'].apply(convert_relative_time)

    # remove duplicates
    df.drop_duplicates(subset=['Text', 'url'], inplace=True)

    # domain column (VERY IMPORTANT)
    df['domain'] = df['source'].map({
        'youtube': 'media',
        'reddit': 'community',
        'googleplay': 'apps',
        'steam': 'gaming',
        'amazon': 'products'
    })

    df.to_csv("Checkpoint1.csv", index=False)

    print(f"\n✅ Extraction complete. {len(df)} rows saved to Checkpoint1.csv")
else:
    print("\n❌ No data collected.")


Choose source:
1. YouTube | 2. Reddit | 3. Google Play | 4. Steam | 5. Amazon

Choose source:
1. YouTube | 2. Reddit | 3. Google Play | 4. Steam | 5. Amazon

Choose source:
1. YouTube | 2. Reddit | 3. Google Play | 4. Steam | 5. Amazon

Choose source:
1. YouTube | 2. Reddit | 3. Google Play | 4. Steam | 5. Amazon

Choose source:
1. YouTube | 2. Reddit | 3. Google Play | 4. Steam | 5. Amazon

Choose source:
1. YouTube | 2. Reddit | 3. Google Play | 4. Steam | 5. Amazon

Choose source:
1. YouTube | 2. Reddit | 3. Google Play | 4. Steam | 5. Amazon

Choose source:
1. YouTube | 2. Reddit | 3. Google Play | 4. Steam | 5. Amazon

Choose source:
1. YouTube | 2. Reddit | 3. Google Play | 4. Steam | 5. Amazon

Choose source:
1. YouTube | 2. Reddit | 3. Google Play | 4. Steam | 5. Amazon

Choose source:
1. YouTube | 2. Reddit | 3. Google Play | 4. Steam | 5. Amazon

Choose source:
1. YouTube | 2. Reddit | 3. Google Play | 4. Steam | 5. Amazon

Choose source:
1. YouTube | 2. Reddit | 3. Google P

In [2]:
df['category'].value_counts()

category
zepto        2224
blinkit      1799
instamart    1425
Name: count, dtype: int64

In [1]:
import pandas as pd
df=pd.read_csv(r"C:\Users\Prabhat\Desktop\Carear\projects\Auto snetiment analyser\Checkpoint1.csv")

In [2]:
df

,url,author,Text,date,source,category,parsed_date,domain
0,googleplay_com.zeptoconsumerapp,Nikita Vishwakarma,very poor experience with zepto. the items del...,2026-04-27 17:00:13,googleplay,zepto,2026-04-27 17:00:13.000000,apps
1,googleplay_com.zeptoconsumerapp,Purvdeep K khachar,no privacy Hamari personal detail leak Hoti ha...,2026-04-27 16:20:02,googleplay,zepto,2026-04-27 16:20:02.000000,apps
2,googleplay_com.zeptoconsumerapp,Sultana Khatoon,this app delivers very slow his one minute mea...,2026-04-27 16:18:05,googleplay,zepto,2026-04-27 16:18:05.000000,apps
3,googleplay_com.zeptoconsumerapp,Ramya Kishoree,give damage or opened package. customer suppor...,2026-04-27 16:05:01,googleplay,zepto,2026-04-27 16:05:01.000000,apps
4,googleplay_com.zeptoconsumerapp,Radhika Kairwan,Very disappointed with this repeated issue. Th...,2026-04-27 16:02:28,googleplay,zepto,2026-04-27 16:02:28.000000,apps
...,...,...,...,...,...,...,...,...
5443,https://www.youtube.com/watch?v=JD15Qt5Kvbw,@anushkakumar9056,so true\ni really miss that too:),3 months ago,youtube,instamart,2026-01-28 17:19:18.875705,media
5444,https://www.youtube.com/watch?v=JD15Qt5Kvbw,@pre_priya,His life update videos are such a vibe .... ❤❤❤,3 months ago,youtube,instamart,2026-01-28 17:19:18.875705,media
5445,https://www.youtube.com/watch?v=JD15Qt5Kvbw,@WhyBhanshu,Cheers 🥂,3 months ago,youtube,instamart,2026-01-28 17:19:18.875705,media
5446,https://www.youtube.com/watch?v=JD15Qt5Kvbw,@WhyBhanshu,"Hahaha, best comment lol :face-red-heart-shape:",3 months ago,youtube,instamart,2026-01-28 17:19:18.875705,media


In [3]:
df.drop_duplicates(inplace=True)

In [4]:
import re
from nltk.tokenize import word_tokenize

def clean(row):
    row = str(row).lower()
    row = re.sub(r'https\S+|www.\S+', "", row)
    row = re.sub(r"&lt;|&gt;|&amp;", " ", row)
    row = row.strip()

    words = word_tokenize(row)
    if len(words) == 0:   # less aggressive
        return None
    return row

df['Text'] = df['Text'].apply(clean)
df.dropna(inplace=True)

In [5]:
import torch
import transformers
from sentence_transformers import SentenceTransformer

print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA:", torch.cuda.is_available())

c:\ai_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Torch: 2.5.1+cu121
Transformers: 4.40.2
CUDA: True


In [6]:
import os
import time
import requests
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
# ----------------------------
# DEVICE SETUP
# ----------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

if device == "cuda":
    torch.backends.cudnn.benchmark = True
# ----------------------------
# SENTIMENT MODEL SETUP
# ----------------------------
model_name_sent = "cardiffnlp/twitter-roberta-base-sentiment-latest"
encoder = AutoTokenizer.from_pretrained(model_name_sent)
sentiment_model = AutoModelForSequenceClassification.from_pretrained(model_name_sent).to(device)
sentiment_model.eval()
# ----------------------------
# IRONY MODEL SETUP
# ----------------------------
model_name_irony = "cardiffnlp/twitter-roberta-base-irony"
irony_tokenizer = AutoTokenizer.from_pretrained(model_name_irony)
model_irony = AutoModelForSequenceClassification.from_pretrained(model_name_irony).to(device)
model_irony.eval()

irony_model = pipeline(
    "text-classification",
    model=model_irony,
    tokenizer=irony_tokenizer,
    device=0 if device == "cuda" else -1
)

Using device: cuda


c:\ai_env\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [7]:

import torch
print(torch.cuda.get_device_name(0))

NVIDIA GeForce RTX 4050 Laptop GPU


In [8]:

# ----------------------------
# HF LLM SETUP
# ----------------------------
import requests

def query(prompt):
    try:
        response = requests.post(
            "http://localhost:11434/api/generate",
            json={
                "model": "qwen2.5:7b",
                "prompt": prompt,
                "stream": False
            },
            timeout=60
        )

        data = response.json()
        return data.get("response", None)

    except Exception as e:
        print("Ollama error:", e)
        return None
# ----------------------------
# LLM HELPER
# ----------------------------
def build_prompt(text, source, category):
    return f"""
You are analyzing user sentiment.

Context:
- Platform: {source}
- Category: {category}

Comment:
{text}

Task:
Classify sentiment as Positive, Neutral, or Negative ONLY , DO NOT include any other text or any other category.

Also consider sarcasm.

Score should reflect intensity:
- 0 = very negative
- 10 = very positive

Score should be in decimal format.

Return ONLY in this format:
Sentiment: <label>
Score: <0 to 10>
"""

def get_llm_sentiment(text, source, category):
    prompt = build_prompt(text, source, category)

    try:
        output = query(prompt)

        sentiment = "neutral"
        score = 5.0

        for line in output.split("\n"):
            line = line.strip()
            if "Sentiment" in line:
                sentiment = line.split(":")[-1].strip().lower()
            elif "Score" in line:
                raw_score = line.split(":")[-1].strip()
                score = float(raw_score)

        return sentiment, score

    except Exception as e:
        print("LLM error:", e)
        return None, None

# ----------------------------
# MAIN FUNCTION
# ----------------------------
def get_sentiment_and_score(texts, sources, categories, batch=8):
    results = []

    for i in range(0, len(texts), batch):
        batch_texts = list(texts[i:i+batch])
        batch_sources = list(sources[i:i+batch])
        batch_categories = list(categories[i:i+batch])

        encoded = encoder(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=256
        )

        encoded = {k: v.to(device) for k, v in encoded.items()}

        with torch.inference_mode():
            outputs = sentiment_model(**encoded)
            logits = outputs.logits
            probs = F.softmax(logits, dim=1)

        pred_ids = torch.argmax(logits, dim=1)

        id2label = {0: "negative", 1: "neutral", 2: "positive"}
        weights = torch.tensor([2.0, 3.0, 8.0], device=device)

        ratings = (probs * weights).sum(dim=1)

        # sarcasm batch
        irony_outputs = irony_model(batch_texts, truncation=True, max_length=256)
        sarcasm_scores = torch.tensor([
        o["score"] if o["label"] == "irony" else 0.0
        for o in irony_outputs
        ], device=device)

        ratings = ratings - 0.75*sarcasm_scores
        ratings = torch.clamp(ratings, 0, 10)

        # -------- LLM fallback logic --------
        for txt, pid, rating, src, cat, sarcasm in zip(
            batch_texts, pred_ids, ratings, batch_sources, batch_categories, sarcasm_scores
        ):
            sentiment = id2label[int(pid.item())]
            final_rating = float(rating.item())
            sarcasm_value = float(sarcasm.item())

            use_llm = False

            # trigger conditions
            if sentiment == "neutral":
                use_llm = True
            if sarcasm_value > 0.7:
                use_llm = True

            # -------- LLM CALL --------
            if use_llm:
                llm_sent, llm_score = get_llm_sentiment(txt, src, cat)

                if llm_sent is not None:
                    sentiment = llm_sent
                    final_rating = llm_score

            # -------- platform weighting --------
            platform_weight = {
                "youtube": 1,
                "reddit": 1,
                "steam": 1,
                "googleplay": 1,
                "amazon": 1
            }

            final_rating = final_rating * platform_weight.get(src, 1.0)
            final_rating = max(0, min(10, final_rating))

            results.append({
                "Text": txt,
                "Sentiment": sentiment,
                "Rating": final_rating,
            })

    return results

In [9]:
len(df[df['source'] == 'googleplay'])

4573

In [10]:
temp = get_sentiment_and_score(
    df['Text'].tolist(),
    df['source'].tolist(),
    df['category'].tolist()
)
temp = pd.DataFrame(temp)
# add ID based on df (NOT data)
df = df.reset_index(drop=True)
temp = temp.reset_index(drop=True)

df['ID'] = df.index
temp['ID'] = temp.index

# merge properly
new_data = pd.merge(df, temp, on='ID', how='inner')
# cleanup
new_data.drop(columns=['Text_x'], inplace=True)
new_data.rename(columns={'Text_y': 'text'}, inplace=True)
new_data['Sentiment'] = new_data['Sentiment'].astype(str).str.lower()

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [11]:
new_data.to_csv("Checkpoint2")

In [12]:
import pandas as pd
import numpy as np
new_data=pd.read_csv(r"C:\Users\Prabhat\Desktop\Carear\projects\Auto snetiment analyser\Checkpoint2")

In [13]:
new_data

,Unnamed: 0,url,author,date,source,category,parsed_date,domain,ID,text,Sentiment,Rating
0,0,googleplay_com.zeptoconsumerapp,Nikita Vishwakarma,2026-04-27 17:00:13,googleplay,zepto,2026-04-27 17:00:13.000000,apps,0,very poor experience with zepto. the items del...,negative,2.087321
1,1,googleplay_com.zeptoconsumerapp,Purvdeep K khachar,2026-04-27 16:20:02,googleplay,zepto,2026-04-27 16:20:02.000000,apps,1,no privacy hamari personal detail leak hoti ha...,negative,2.500000
2,2,googleplay_com.zeptoconsumerapp,Sultana Khatoon,2026-04-27 16:18:05,googleplay,zepto,2026-04-27 16:18:05.000000,apps,2,this app delivers very slow his one minute mea...,negative,2.084432
3,3,googleplay_com.zeptoconsumerapp,Ramya Kishoree,2026-04-27 16:05:01,googleplay,zepto,2026-04-27 16:05:01.000000,apps,3,give damage or opened package. customer suppor...,negative,2.141303
4,4,googleplay_com.zeptoconsumerapp,Radhika Kairwan,2026-04-27 16:02:28,googleplay,zepto,2026-04-27 16:02:28.000000,apps,4,very disappointed with this repeated issue. th...,negative,2.132904
...,...,...,...,...,...,...,...,...,...,...,...,...
5442,5442,https://www.youtube.com/watch?v=JD15Qt5Kvbw,@anushkakumar9056,3 months ago,youtube,instamart,2026-01-28 17:19:18.875705,media,5442,so true\ni really miss that too:),positive,7.754591
5443,5443,https://www.youtube.com/watch?v=JD15Qt5Kvbw,@pre_priya,3 months ago,youtube,instamart,2026-01-28 17:19:18.875705,media,5443,his life update videos are such a vibe .... ❤❤❤,positive,7.527386
5444,5444,https://www.youtube.com/watch?v=JD15Qt5Kvbw,@WhyBhanshu,3 months ago,youtube,instamart,2026-01-28 17:19:18.875705,media,5444,cheers 🥂,positive,7.343755
5445,5445,https://www.youtube.com/watch?v=JD15Qt5Kvbw,@WhyBhanshu,3 months ago,youtube,instamart,2026-01-28 17:19:18.875705,media,5445,"hahaha, best comment lol :face-red-heart-shape:",positive,7.839929


In [14]:
new_data = new_data.reset_index(drop=True)
new_data['ID'] = new_data.index

In [15]:
import pandas as pd
import numpy as np
import torch
import umap
import hdbscan
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
import math
# ----------------------------
# DEVICE SETUP
# ----------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# ----------------------------
# TRANSFORMER SETUP
# ----------------------------
transformer = SentenceTransformer("all-roberta-large-v1", device=device)

# ----------------------------
# TOPIC QUALITY HELPERS
# ----------------------------
def topic_diversity(topic_keywords, top_k=10):
    words = []
    for topic_id, kw in topic_keywords.items():
        if topic_id == -1 or kw == "Misc / Noise":
            continue
        parts = [w.strip() for w in kw.split(",")[:top_k] if w.strip()]
        words.extend(parts)

    if len(words) == 0:
        return np.nan

    return len(set(words)) / len(words)


def safe_topic_quality(cluster_labels, probabilities, topic_keywords):
    labels = np.asarray(cluster_labels)
    mask = labels != -1  # ignore noise

    if mask.sum() == 0:
        return np.nan, np.nan, np.nan

    if probabilities is not None and len(probabilities) == len(labels):
        avg_confidence = float(np.mean(probabilities[mask]))
    else:
        avg_confidence = np.nan

    diversity = topic_diversity(topic_keywords)

    if np.isnan(avg_confidence) and np.isnan(diversity):
        quality = np.nan
    elif np.isnan(avg_confidence):
        quality = diversity
    elif np.isnan(diversity):
        quality = avg_confidence
    else:
        quality = 0.5 * avg_confidence + 0.5 * diversity

    return (
        float(quality),
        float(avg_confidence) if not np.isnan(avg_confidence) else np.nan,
        float(diversity) if not np.isnan(diversity) else np.nan
    )

# ----------------------------
# TOPIC EXTRACTION FUNCTION
# ----------------------------
def build_topics(df, text_col="text", topic_col="Topic", keyword_col="Topic_Keywords"):
    n_rows=len(df)
    out = df.copy()
    out[text_col] = out[text_col].fillna("").astype(str)

    texts = out[text_col].tolist()
    n = len(texts)

    if n == 0:
        out[topic_col] = []
        out[keyword_col] = []
        out["Topic_Quality_Score"] = np.nan
        out["Cluster_Confidence"] = np.nan
        out["Topic_Diversity"] = np.nan
        return out, np.nan

    embeddings = transformer.encode(
        texts,
        batch_size=64 if device == "cuda" else 32,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    if n < 5:
        out[topic_col] = 0
        out[keyword_col] = "Misc / Noise"
        out["Cluster_Confidence"] = np.nan
        out["Topic_Diversity"] = np.nan
        out["Topic_Quality_Score"] = np.nan
        return out, np.nan

    n_neighbors = int(min(50, max(12, round(math.sqrt(n_rows) * 1.2))))
    n_components = 10 if n_rows >= 1200 else 8

    n_components = max(2, min(7, n - 1))

    umap_model = umap.UMAP(
        n_neighbors=n_neighbors,
        n_components=n_components,
        metric="cosine",
        random_state=42
    )
    umap_embeddings = umap_model.fit_transform(embeddings)

    min_cluster_size = int(min(30, max(6, round(n_rows * 0.015))))
    min_samples = int(min(6, max(2, round(min_cluster_size * 0.35))))

    hdb_model = hdbscan.HDBSCAN(
        min_cluster_size=min_cluster_size,
        min_samples=min_samples,
        metric="euclidean",
        cluster_selection_method="eom"
    )

    cluster_labels = hdb_model.fit_predict(umap_embeddings)

    # Build keyword summary per topic
    temp = pd.DataFrame({
        "text": out[text_col].values,
        "topic": cluster_labels
    })

    topic_text_map = {}
    for topic_id in temp["topic"].unique():
        joined_text = " ".join(temp[temp["topic"] == topic_id]["text"].tolist())
        topic_text_map[topic_id] = joined_text

    topic_df = pd.DataFrame({
        "topic": list(topic_text_map.keys()),
        "text": list(topic_text_map.values())
    })

    vectorizer = TfidfVectorizer(stop_words="english")

    if len(topic_df) > 0 and topic_df["text"].str.strip().any():
        matrix_tfidf = vectorizer.fit_transform(topic_df["text"]).toarray()
        feature_names = vectorizer.get_feature_names_out()

        topic_keywords = {}
        for topic_id, row in zip(topic_df["topic"], matrix_tfidf):
            if topic_id == -1:
                topic_keywords[topic_id] = "Misc / Noise"
                continue

            top_idx = np.argsort(row)[-20:][::-1]
            top_words = [feature_names[i] for i in top_idx if row[i] > 0]
            if len(top_words) == 0:
                topic_keywords[topic_id] = "Misc / Noise"
            else:
                topic_keywords[topic_id] = ", ".join(top_words[:20])
    else:
        topic_keywords = {t: "Misc / Noise" for t in temp["topic"].unique()}

    # New quality metric
    topic_quality, avg_confidence, diversity = safe_topic_quality(
        cluster_labels,
        getattr(hdb_model, "probabilities_", None),
        topic_keywords
    )

    out[topic_col] = cluster_labels
    out[keyword_col] = out[topic_col].map(topic_keywords).fillna("Misc / Noise")

    probs = getattr(hdb_model, "probabilities_", None)
    if probs is not None and len(probs) == len(out):
        out["Cluster_Confidence"] = probs
    else:
        out["Cluster_Confidence"] = np.nan

    out["Topic_Diversity"] = diversity

    return out, topic_quality

# ----------------------------
# BUILD LOCAL + GLOBAL TOPICS
# ----------------------------
def build_complete_dataset(new_data):
    local_parts = []
    validation_rows = []

    # Local topics per category
    for category in new_data["category"].dropna().unique():
        subdf = new_data[new_data["category"] == category].copy()

        local_df, local_score = build_topics(
            subdf,
            text_col="text",
            topic_col="Local_Topic",
            keyword_col="Local_Topic_Keywords"
        )

        local_df["Local_Topic_Quality"] = local_score
        local_parts.append(local_df)

        validation_rows.append({
            "Scope": "Local",
            "Category": category,
            "Topic_Quality_Score": local_score,
            "Rows": len(local_df),
            "Num_Topics": local_df["Local_Topic"].nunique()
        })

    local_complete = pd.concat(local_parts, ignore_index=True) if local_parts else pd.DataFrame()

    # Global topics across all data
    global_complete, global_score = build_topics(
        new_data.copy(),
        text_col="text",
        topic_col="Global_Topic",
        keyword_col="Global_Topic_Keywords"
    )

    global_complete["Global_Topic_Quality"] = global_score

    validation_rows.append({
        "Scope": "Global",
        "Category": "ALL",
        "Topic_Quality_Score": global_score,
        "Rows": len(global_complete),
        "Num_Topics": global_complete["Global_Topic"].nunique()
    })

    # Align and merge global topics into local dataset
    global_complete = global_complete.reset_index(drop=True)
    local_complete = local_complete.reset_index(drop=True)

    complete_df = pd.merge(
        local_complete,
        global_complete[[
            "ID",
            "Global_Topic",
            "Global_Topic_Keywords",
            "Global_Topic_Quality",
            "Cluster_Confidence",
            "Topic_Diversity"
        ]].rename(columns={
            "Cluster_Confidence": "Global_Cluster_Confidence",
            "Topic_Diversity": "Global_Topic_Diversity"
        }),
        on="ID",
        how="left"
    )
    validation_df = pd.DataFrame(validation_rows)

    return complete_df, validation_df

# ----------------------------
# USAGE
# ----------------------------
complete_dataset, validation_df = build_complete_dataset(new_data)

Using device: cuda


c:\ai_env\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Batches: 100%|██████████| 35/35 [00:22<00:00,  1.58it/s]
c:\ai_env\lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
Batches: 100%|██████████| 29/29 [00:17<00:00,  1.66it/s]
c:\ai_env\lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
Batches: 100%|██████████| 23/23 [00:12<00:00,  1.91it/s]
c:\ai_env\lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
Batches: 100%|██████████| 86/86 [00:50<00:00,  1.70it/s]
c:\ai_env\lib\site-packages\umap\umap_.

In [16]:

complete_dataset.to_csv(f"complete_data.csv", index=False) # checkpoint 3

In [17]:
import pandas as pd
complete_dataset=pd.read_csv(r"C:\Users\Prabhat\Desktop\Carear\projects\Auto snetiment analyser\complete_data.csv")
complete_dataset

,Unnamed: 0,url,author,date,source,category,parsed_date,domain,ID,text,...,Local_Topic,Local_Topic_Keywords,Cluster_Confidence,Topic_Diversity,Local_Topic_Quality,Global_Topic,Global_Topic_Keywords,Global_Topic_Quality,Global_Cluster_Confidence,Global_Topic_Diversity
0,0,googleplay_com.zeptoconsumerapp,Nikita Vishwakarma,2026-04-27 17:00:13,googleplay,zepto,2026-04-27 17:00:13.000000,apps,0,very poor experience with zepto. the items del...,...,3,"zepto, order, delivery, customer, app, service...",1.000000,0.700000,0.790351,0,"zepto, delivery, order, customer, app, product...",0.778732,1.000000,0.645946
1,1,googleplay_com.zeptoconsumerapp,Purvdeep K khachar,2026-04-27 16:20:02,googleplay,zepto,2026-04-27 16:20:02.000000,apps,1,no privacy hamari personal detail leak hoti ha...,...,1,"hai, bhi, nahi, se, ke, nhi, ki, koi, ka, aur,...",0.810939,0.700000,0.790351,24,"app, hai, bhi, bahut, download, hi, mat, india...",0.778732,1.000000,0.645946
2,2,googleplay_com.zeptoconsumerapp,Sultana Khatoon,2026-04-27 16:18:05,googleplay,zepto,2026-04-27 16:18:05.000000,apps,2,this app delivers very slow his one minute mea...,...,6,"app, order, delivery, customer, worst, service...",1.000000,0.700000,0.790351,-1,Misc / Noise,0.778732,0.000000,0.645946
3,3,googleplay_com.zeptoconsumerapp,Ramya Kishoree,2026-04-27 16:05:01,googleplay,zepto,2026-04-27 16:05:01.000000,apps,3,give damage or opened package. customer suppor...,...,6,"app, order, delivery, customer, worst, service...",1.000000,0.700000,0.790351,33,"customer, support, product, return, ordered, i...",0.778732,1.000000,0.645946
4,4,googleplay_com.zeptoconsumerapp,Radhika Kairwan,2026-04-27 16:02:28,googleplay,zepto,2026-04-27 16:02:28.000000,apps,4,very disappointed with this repeated issue. th...,...,6,"app, order, delivery, customer, worst, service...",1.000000,0.700000,0.790351,34,"milk, expiry, expired, product, products, deli...",0.778732,1.000000,0.645946
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5442,5442,https://www.youtube.com/watch?v=JD15Qt5Kvbw,@anushkakumar9056,3 months ago,youtube,instamart,2026-01-28 17:19:18.875705,media,5442,so true\ni really miss that too:),...,2,"video, videos, content, just, like, really, lo...",0.509057,0.966667,0.970609,-1,Misc / Noise,0.778732,0.000000,0.645946
5443,5443,https://www.youtube.com/watch?v=JD15Qt5Kvbw,@pre_priya,3 months ago,youtube,instamart,2026-01-28 17:19:18.875705,media,5443,his life update videos are such a vibe .... ❤❤❤,...,2,"video, videos, content, just, like, really, lo...",0.488992,0.966667,0.970609,6,"videos, video, content, love, research, consul...",0.778732,1.000000,0.645946
5444,5444,https://www.youtube.com/watch?v=JD15Qt5Kvbw,@WhyBhanshu,3 months ago,youtube,instamart,2026-01-28 17:19:18.875705,media,5444,cheers 🥂,...,1,"good, nice, super, best, ok, excellent, bad, g...",1.000000,0.966667,0.970609,7,"nice, red, heart, thank, thj5quraqrfypcblu4, y...",0.778732,0.245941,0.645946
5445,5445,https://www.youtube.com/watch?v=JD15Qt5Kvbw,@WhyBhanshu,3 months ago,youtube,instamart,2026-01-28 17:19:18.875705,media,5445,"hahaha, best comment lol :face-red-heart-shape:",...,1,"good, nice, super, best, ok, excellent, bad, g...",0.974263,0.966667,0.970609,7,"nice, red, heart, thank, thj5quraqrfypcblu4, y...",0.778732,0.190705,0.645946


In [18]:
for val in complete_dataset['category'].unique():
    print(complete_dataset[complete_dataset['category'] == val ]['Local_Topic'].nunique(), val)
print(complete_dataset['Global_Topic'].nunique())


9 zepto
14 blinkit
3 instamart
38


In [19]:
import requests
import pandas as pd
import json
import re

# ----------------------------
# OLLAMA SETUP
# ----------------------------
OLLAMA_URL = "http://localhost:11434/api/generate"
OLLAMA_MODEL = "qwen2.5:7b"

# ----------------------------
# QUERY FUNCTION
# ----------------------------
def query(prompt):
    try:
        resp = requests.post(
            OLLAMA_URL,
            json={
                "model": OLLAMA_MODEL,
                "prompt": prompt,
                "stream": False,
                "options": {
                    "temperature": 0.2
                }
            },
            timeout=120
        )

        if resp.status_code != 200:
            print("Bad response:", resp.status_code)
            return None

        data = resp.json()
        return data.get("response", None)

    except Exception as e:
        print("Ollama error:", e)
        return None


# ----------------------------
# SAFE JSON PARSER
# ----------------------------
def safe_json_parse(text):
    if not text or not isinstance(text, str):
        return None

    text = text.strip()

    # try direct
    try:
        return json.loads(text)
    except:
        pass

    # 🔥 extract JSON block more aggressively
    match = re.search(r'\{.*\}', text, re.DOTALL)
    if match:
        cleaned = match.group()

        # fix common issues
        cleaned = cleaned.replace("\n", " ")
        cleaned = re.sub(r',\s*}', '}', cleaned)  # remove trailing commas

        try:
            return json.loads(cleaned)
        except:
            return None

    return None

# ----------------------------
# GENERATE NAMES FUNCTION
# ----------------------------
def generate_topic_names(df, topic_col, keyword_col, is_global=False):
    results = []

    # grouping logic
    group_cols = [topic_col] if is_global else [topic_col, "category"]

    for group_vals, subset in df.groupby(group_cols):

        # 🔥 SAFE UNPACKING (fixes tuple bug)
        if isinstance(group_vals, tuple):
            if is_global:
                topic_id = group_vals[0]
                category = "GLOBAL"
            else:
                topic_id, category = group_vals
        else:
            topic_id = group_vals
            category = "GLOBAL" if is_global else None

        # normalize
        try:
            topic_id = int(topic_id)
        except:
            print("Invalid topic_id:", topic_id)
            continue

        if topic_id == -1:
            continue

        if subset.empty:
            continue

        category = str(category).strip()

        keywords_series = subset[keyword_col].dropna()
        if keywords_series.empty:
            continue

        keywords = (
    keywords_series
    .astype(str)
    .str.split(",")
    .explode()
    .str.strip()
    .value_counts()
    .head(20)
    .index.tolist()
)

        filtered_texts = (
    subset["text"]
    .dropna()
    .astype(str)
)

# 🔥 apply filter safely
        filtered_texts = filtered_texts[filtered_texts.str.len() > 40]

        # fallback if too few texts
        if len(filtered_texts) < 3:
            filtered_texts = subset["text"].dropna().astype(str)

        sample_size = min(8, len(filtered_texts))

        sample_texts = (
            filtered_texts
            .sample(n=sample_size, random_state=42)
            .tolist()
        )

        prompt = f"""
You are a strict business topic naming system.

Your job:
Generate a precise, meaningful topic name using the keywords and texts.

IMPORTANT RULES:
- You MUST generate a meaningful name
- Focus on the SINGLE most dominant issue. Never output placeholders like "category Topic X".
- DO NOT repeat the category name
- The name should reflect the MAIN ISSUE or THEME
- Return ONLY a valid JSON object. "Do not include any text before or after the JSON" The response MUST start with '{' and end with '}'

BAD examples:
- Topic 7
- General Feedback
- Miscellaneous

Output format:
{{
  "topic_id": {topic_id},
  "category_name": "{category}",
  "topic_name": "<<your_topic_here>>"
}}
Inputs-:

Category:
{category}

Keywords:
{keywords}

Sample texts:
{sample_texts}
"""

        try:
            result = query(prompt)

            if result is None:
                print("Empty response:", topic_id, category)
                continue

            parsed = safe_json_parse(result)

            if parsed:
                parsed["scope"] = "global" if is_global else "local"
                results.append(parsed)
            else:
                print("Parse failed:", topic_id, category)

        except Exception as e:
            print("Error:", topic_id, category, e)

    return results


# ----------------------------
# RUN FOR BOTH LEVELS
# ----------------------------
local_names = generate_topic_names(
    complete_dataset,
    topic_col="Local_Topic",
    keyword_col="Local_Topic_Keywords",
    is_global=False
)

global_names = generate_topic_names(
    complete_dataset,
    topic_col="Global_Topic",
    keyword_col="Global_Topic_Keywords",
    is_global=True
)

# ----------------------------
# COMBINE + CONVERT
# ----------------------------
names = local_names + global_names
names_df = pd.DataFrame(names)

print("\nSample Output:\n", names_df.head())
print("\nTotal generated:", len(names_df))


Sample Output:
    topic_id category_name                                         topic_name  \
0         0       blinkit           User Feedback on Quality and Suggestions   
1         0     instamart                Delivery Issues and Refund Problems   
2         0         zepto  Digital Marketing Skills and Freelancing Oppor...   
3         1       blinkit                                 Experience Quality   
4         1     instamart                              Moderator Recruitment   

   scope  
0  local  
1  local  
2  local  
3  local  
4  local  

Total generated: 61


In [20]:
print(names_df)

    topic_id category_name                                         topic_name  \
0          0       blinkit           User Feedback on Quality and Suggestions   
1          0     instamart                Delivery Issues and Refund Problems   
2          0         zepto  Digital Marketing Skills and Freelancing Oppor...   
3          1       blinkit                                 Experience Quality   
4          1     instamart                              Moderator Recruitment   
..       ...           ...                                                ...   
56        32        GLOBAL            Delivery Time Accuracy and Transparency   
57        33        GLOBAL                            Customer Support Issues   
58        34        GLOBAL                       Expired and Damaged Products   
59        35        GLOBAL         Worst Customer Service and Delivery Issues   
60        36        GLOBAL                              Poor Customer Service   

     scope  
0    local  
1

In [21]:
names_df['topic_name'].nunique()

58

In [22]:
# ----------------------------
# USE CLEAN NAMES
# ----------------------------
names_list = local_names + global_names
names_df = pd.DataFrame(names_list).copy()

# ----------------------------
# GENERALIZATION
# ----------------------------
import json
import re
import requests
import pandas as pd

OLLAMA_URL = "http://localhost:11434/api/generate"
OLLAMA_MODEL = "qwen2.5:7b"

# ----------------------------
# SAFE JSON PARSER
# ----------------------------
def safe_json_parse(text):
    if not text or not isinstance(text, str):
        return None

    text = text.strip()

    # direct parse first
    try:
        return json.loads(text)
    except:
        pass

    # fallback: extract first JSON array block
    matches = re.findall(r'\[.*?\]', text, re.DOTALL)
    for m in matches:
        try:
            return json.loads(m)
        except:
            continue

    return None


# ----------------------------
# QUERY FUNCTION
# ----------------------------
def query_generalization(category, topics, scope="mixed"):
    topics_text = json.dumps(topics, ensure_ascii=False)

    prompt = f"""
You are a business topic consolidation system.

Task:
Merge overlapping or very similar topic names into broader business topics.

Rules:
- Output ONLY a valid JSON array
- No explanations, no markdown
- If no merging is needed, return []
- Keep "old" EXACTLY as given
- Keep names short and business-friendly
- DO NOT include any explanatory text or formatting outside the JSON object

Each output item MUST be:
{{
  "category_name": "{category}",
  "scope": "{scope}",
  "old": "exact original topic",
  "new": "generalized topic"
}}

Example:
Input:
["Late delivery", "Delayed shipping"]

Output:
[
  {{
    "category_name": "{category}",
    "scope": "{scope}",
    "old": "Late delivery",
    "new": "Delivery Delay Issues"
  }},
  {{
    "category_name": "{category}",
    "scope": "{scope}",
    "old": "Delayed shipping",
    "new": "Delivery Delay Issues"
  }}
]

Topics:
{topics_text}
"""

    try:
        resp = requests.post(
            OLLAMA_URL,
            json={
                "model": OLLAMA_MODEL,
                "prompt": prompt,
                "stream": False,
                "options": {
                    "temperature": 0.2
                }
            },
            timeout=120
        )

        if resp.status_code != 200:
            print(f"[{category} | {scope}] Bad response:", resp.status_code)
            return None

        data = resp.json()
        return data.get("response", None)

    except Exception as e:
        print(f"[{category} | {scope}] Ollama error:", e)
        return None


# ----------------------------
# RUN GENERALIZATION
# ----------------------------
subtopics = []

group_cols = ["category_name", "scope"] if "scope" in names_df.columns else ["category_name"]

for group_vals, subset in names_df.groupby(group_cols, dropna=False):

    # safe unpacking for both single-key and multi-key groupby results
    if isinstance(group_vals, tuple):
        if len(group_vals) == 2:
            category, scope = group_vals
        elif len(group_vals) == 1:
            category = group_vals[0]
            scope = "mixed"
        else:
            category = str(group_vals[0]) if len(group_vals) > 0 else "unknown"
            scope = "mixed"
    else:
        category = group_vals
        scope = "mixed"

    category = str(category).strip().lower()
    scope = str(scope).strip().lower()

    topics = (
        subset["topic_name"]
        .dropna()
        .astype(str)
        .str.strip()
        .unique()
        .tolist()
    )

    if len(topics) < 2:
        continue

    # keep prompt stable
    topics = sorted(topics)

    result = query_generalization(category, topics, scope=scope)

    print(f"\n--- RAW OUTPUT ({category} | {scope}) ---")
    print(result)

    if result is None:
        print(f"Parse failed for category: {category} | scope: {scope}")
        continue

    if result.strip() == "[]":
        continue

    parsed = safe_json_parse(result)

    if parsed and isinstance(parsed, list):
        subtopics.extend(parsed)
    else:
        print(f"Parse failed for category: {category} | scope: {scope}")


# ----------------------------
# CLEAN OUTPUT
# ----------------------------
generalized_df = pd.DataFrame(subtopics)

if generalized_df.empty:
    generalized_df = pd.DataFrame(columns=["category_name", "scope", "old", "new"])

generalized_df = generalized_df.drop_duplicates()

print("\nSample:")
print(generalized_df.head())
print("\nTotal:", len(generalized_df))


--- RAW OUTPUT (global | global) ---
[
  {
    "category_name": "global",
    "scope": "global",
    "old": "App Quality and User Experience Issues",
    "new": "User Experience Issues"
  },
  {
    "category_name": "global",
    "scope": "global",
    "old": "Customer Satisfaction",
    "new": "Customer Feedback"
  },
  {
    "category_name": "global",
    "scope": "global",
    "old": "Customer Support Issues",
    "new": "Customer Service Issues"
  },
  {
    "category_name": "global",
    "scope": "global",
    "old": "Declining Service Quality",
    "new": "Service Quality Issues"
  },
  {
    "category_name": "global",
    "scope": "global",
    "old": "Delivery Partner Behavior and Service",
    "new": "Delivery Service Issues"
  },
  {
    "category_name": "global",
    "scope": "global",
    "old": "Delivery Time Accuracy and Transparency",
    "new": "Delivery Timeliness Issues"
  },
  {
    "category_name": "global",
    "scope": "global",
    "old": "Experience Quality",
 

In [23]:
generalized_df['old'].unique()

array(['App Quality and User Experience Issues', 'Customer Satisfaction',
       'Customer Support Issues', 'Declining Service Quality',
       'Delivery Partner Behavior and Service',
       'Delivery Time Accuracy and Transparency', 'Experience Quality',
       'Fast and Quality Delivery Services',
       'Fast and Reliable Grocery Delivery Services',
       'High Delivery and Surcharge Fees',
       'Instamart Delivery and Refund Issues',
       'Legal Issues and Brand Scrutiny', 'Location Accuracy Issues',
       'Melted Ice Cream Delivery Issues',
       'Order Issues and Pricing Concerns',
       'Petrol Costs and Travel Expenses', 'Poor Customer Service',
       'Positive Feedback and Experiences',
       'Quality and Freshness Issues', 'Quick_and_Fast_Delivery',
       'Refund Issues and Customer Service',
       'Refund Issues and Misdelivered Products', 'Service Quality',
       'Very Positive Feedback', 'Worst Apps and Applications',
       'Worst Customer Service and Delive

In [24]:
from merge_topic_labels import merge_topic_labels
merged, local_map, global_map, diagnostics = merge_topic_labels(
    complete_dataset, local_names, global_names, generalized_df
)
print(diagnostics)


{'missing_local_mapping_before_fallback': 0, 'missing_global_mapping_before_fallback': 0, 'dataset_local_keys': 26, 'dataset_global_keys': 38, 'named_local_keys': 24, 'named_global_keys': 37}


In [25]:
print(merged[["local_topic_name","global_topic_name"]].isna().sum())


local_topic_name     0
global_topic_name    0
dtype: int64


In [26]:
len(merged[merged['global_topic_name'] == -1])

0

In [27]:
local_map.duplicated(subset=["category_name", "topic_id"]).sum()

np.int64(0)

In [28]:
from scipy.stats import f_oneway, ttest_ind

platforms = merged['source'].dropna().unique()
print("Platforms:", platforms)

# Filter valid groups
valid_groups = {}
for p in platforms:
    group = merged[merged['source'] == p]['Rating']
    if len(group) > 5:   # minimum size
        valid_groups[p] = group

num_groups = len(valid_groups)

if num_groups < 2:
    print("❌ Not enough valid groups for statistical testing")

elif num_groups == 2:
    p1, p2 = list(valid_groups.keys())

    g1 = valid_groups[p1]
    g2 = valid_groups[p2]

    stat, p = ttest_ind(g1, g2, equal_var=False)

    print(f"\nT-test between {p1} and {p2}")
    print("p-value:", p)

else:
    stat, p = f_oneway(*valid_groups.values())

    print("\nANOVA across platforms")
    print("Groups used:", list(valid_groups.keys()))
    print("p-value:", p)
    
if num_groups >= 2:
    if p < 0.05:
        print("✅ Significant difference detected")
    else:
        print("❌ No significant difference detected")

Platforms: ['googleplay' 'reddit' 'youtube']

ANOVA across platforms
Groups used: ['googleplay', 'reddit', 'youtube']
p-value: 3.3138435161091506e-06
✅ Significant difference detected


In [29]:
merged.drop_duplicates()

,Unnamed: 0,url,author,date,source,category,parsed_date,domain,ID,text,...,Local_Topic_Quality,Global_Topic,Global_Topic_Keywords,Global_Topic_Quality,Global_Cluster_Confidence,Global_Topic_Diversity,local_topic_name,local_generalized_topic,global_topic_name,global_generalized_topic
0,0,googleplay_com.zeptoconsumerapp,Nikita Vishwakarma,2026-04-27 17:00:13,googleplay,zepto,2026-04-27 17:00:13.000000,apps,0,very poor experience with zepto. the items del...,...,0.790351,0,"zepto, delivery, order, customer, app, product...",0.778732,1.000000,0.645946,Delivery Issues and Customer Service,Delivery Issues,Zepto Delivery Issues and Poor Customer Service,Customer Service Issues
1,1,googleplay_com.zeptoconsumerapp,Purvdeep K khachar,2026-04-27 16:20:02,googleplay,zepto,2026-04-27 16:20:02.000000,apps,1,no privacy hamari personal detail leak hoti ha...,...,0.790351,24,"app, hai, bhi, bahut, download, hi, mat, india...",0.778732,1.000000,0.645946,Delivery Issues and Fraud Allegations,Delivery Issues,App Quality and User Experience Issues,User Experience Issues
2,2,googleplay_com.zeptoconsumerapp,Sultana Khatoon,2026-04-27 16:18:05,googleplay,zepto,2026-04-27 16:18:05.000000,apps,2,this app delivers very slow his one minute mea...,...,0.790351,-1,Misc / Noise,0.778732,0.000000,0.645946,Poor Delivery Service and Refund Issues,Delivery Issues,Noise / Unclustered,Noise / Unclustered
3,3,googleplay_com.zeptoconsumerapp,Ramya Kishoree,2026-04-27 16:05:01,googleplay,zepto,2026-04-27 16:05:01.000000,apps,3,give damage or opened package. customer suppor...,...,0.790351,33,"customer, support, product, return, ordered, i...",0.778732,1.000000,0.645946,Poor Delivery Service and Refund Issues,Delivery Issues,Customer Support Issues,Customer Service Issues
4,4,googleplay_com.zeptoconsumerapp,Radhika Kairwan,2026-04-27 16:02:28,googleplay,zepto,2026-04-27 16:02:28.000000,apps,4,very disappointed with this repeated issue. th...,...,0.790351,34,"milk, expiry, expired, product, products, deli...",0.778732,1.000000,0.645946,Poor Delivery Service and Refund Issues,Delivery Issues,Expired and Damaged Products,Expired and Damaged Products
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5442,5442,https://www.youtube.com/watch?v=JD15Qt5Kvbw,@anushkakumar9056,3 months ago,youtube,instamart,2026-01-28 17:19:18.875705,media,5442,so true\ni really miss that too:),...,0.970609,-1,Misc / Noise,0.778732,0.000000,0.645946,User Feedback on Video Content,User Feedback on Video Content,Noise / Unclustered,Noise / Unclustered
5443,5443,https://www.youtube.com/watch?v=JD15Qt5Kvbw,@pre_priya,3 months ago,youtube,instamart,2026-01-28 17:19:18.875705,media,5443,his life update videos are such a vibe .... ❤❤❤,...,0.970609,6,"videos, video, content, love, research, consul...",0.778732,1.000000,0.645946,User Feedback on Video Content,User Feedback on Video Content,Video Content Engagement and Passion,Video Content Engagement and Passion
5444,5444,https://www.youtube.com/watch?v=JD15Qt5Kvbw,@WhyBhanshu,3 months ago,youtube,instamart,2026-01-28 17:19:18.875705,media,5444,cheers 🥂,...,0.970609,7,"nice, red, heart, thank, thj5quraqrfypcblu4, y...",0.778732,0.245941,0.645946,Moderator Recruitment,Moderator Recruitment,Positive Feedback and Emotions,Positive Feedback and Emotions
5445,5445,https://www.youtube.com/watch?v=JD15Qt5Kvbw,@WhyBhanshu,3 months ago,youtube,instamart,2026-01-28 17:19:18.875705,media,5445,"hahaha, best comment lol :face-red-heart-shape:",...,0.970609,7,"nice, red, heart, thank, thj5quraqrfypcblu4, y...",0.778732,0.190705,0.645946,Moderator Recruitment,Moderator Recruitment,Positive Feedback and Emotions,Positive Feedback and Emotions


In [30]:
merged['parsed_date']

0       2026-04-27 17:00:13.000000
1       2026-04-27 16:20:02.000000
2       2026-04-27 16:18:05.000000
3       2026-04-27 16:05:01.000000
4       2026-04-27 16:02:28.000000
                   ...            
5442    2026-01-28 17:19:18.875705
5443    2026-01-28 17:19:18.875705
5444    2026-01-28 17:19:18.875705
5445    2026-01-28 17:19:18.875705
5446    2026-01-28 17:19:18.875705
Name: parsed_date, Length: 5447, dtype: object

In [31]:
for cat in merged['category'].unique():
    temp=merged[merged['category']==cat]
    print( cat , temp['local_topic_name'].nunique())
    print("generalized",cat ,temp['local_generalized_topic'].nunique())
print("global", merged['global_topic_name'].nunique())
print("global", merged['global_generalized_topic'].nunique())

zepto 9
generalized zepto 7
blinkit 14
generalized blinkit 6
instamart 3
generalized instamart 3
global 37
global 28
 9
generalized zepto 7
blinkit 14
generalized blinkit 6
instamart 3
generalized instamart 3
global 37
global 28


In [32]:
merged.to_excel("final_output.xlsx", index=False, engine="xlsxwriter")

In [33]:
# ----------------------------
# UPDATED VALIDATION CHECK
# ----------------------------

# LOCAL
noise_local = (merged["Local_Topic"] == -1).sum()
noise_labeled_local = (merged["local_topic_name"] == "Noise / Unclustered").sum()

print("Local noise:", noise_local)
print("Local labeled as noise:", noise_labeled_local)
print("Match:", noise_local == noise_labeled_local)

# GLOBAL
noise_global = (merged["Global_Topic"] == -1).sum()
noise_labeled_global = (merged["global_topic_name"] == "Noise / Unclustered").sum()

print("Global noise:", noise_global)
print("Global labeled as noise:", noise_labeled_global)
print("Match:", noise_global == noise_labeled_global)

Local noise: 605
Local labeled as noise: 605
Match: True
Global noise: 1442
Global labeled as noise: 1442
Match: True


In [34]:
merged.columns

Index(['Unnamed: 0', 'url', 'author', 'date', 'source', 'category',
       'parsed_date', 'domain', 'ID', 'text', 'Sentiment', 'Rating',
       'Local_Topic', 'Local_Topic_Keywords', 'Cluster_Confidence',
       'Topic_Diversity', 'Local_Topic_Quality', 'Global_Topic',
       'Global_Topic_Keywords', 'Global_Topic_Quality',
       'Global_Cluster_Confidence', 'Global_Topic_Diversity',
       'local_topic_name', 'local_generalized_topic', 'global_topic_name',
       'global_generalized_topic'],
      dtype='object')

In [35]:
merged['category'].unique()

array(['zepto', 'blinkit', 'instamart'], dtype=object)

In [36]:
merged['local_topic_name'].unique()

array(['Delivery Issues and Customer Service',
       'Delivery Issues and Fraud Allegations',
       'Poor Delivery Service and Refund Issues', 'Noise / Unclustered',
       'Positive User Feedback', 'App Quality and Service Issues',
       'FastAndAffordableDelivery',
       'Digital Marketing Skills and Freelancing Opportunities',
       'Parental Communication and Marketing Practices',
       'User Feedback on Quality and Suggestions', 'Experience Quality',
       'Fast Delivery App Reviews',
       'Customer Experience with Delivery and Service',
       'Delivery Issues and Service Concerns',
       'Fast and Reliable Delivery Services',
       'Customer Service and Product Issues',
       'Quality and Service Decline', 'Delivery Timings and Reliability',
       'Delivery Boy Behavior and Timeliness',
       'High Delivery Charges and Surge Pricing',
       'Labourer Compensation and Working Conditions',
       'Delivery Issues and Refund Problems', 'Moderator Recruitment',
      

In [37]:
merged['local_generalized_topic'].unique()

array(['Delivery Issues', 'Noise / Unclustered', 'User Feedback',
       'Service Quality Issues', 'Delivery Speed and Cost',
       'Digital Marketing Skills and Freelancing Opportunities',
       'Marketing Practices', 'Quality and Service Decline',
       'Delivery and Service Issues', 'Service and Quality Concerns',
       'Cost Concerns', 'Workforce Management Issues',
       'Delivery Issues and Refund Problems', 'Moderator Recruitment',
       'User Feedback on Video Content'], dtype=object)

In [38]:
merged['global_topic_name'].unique()

array(['Zepto Delivery Issues and Poor Customer Service',
       'App Quality and User Experience Issues', 'Noise / Unclustered',
       'Customer Support Issues', 'Expired and Damaged Products',
       'Service Quality', 'Fast and Reliable Grocery Delivery Services',
       'Location Accuracy Issues',
       'Delivery Time Accuracy and Transparency',
       'Worst Delivery and Refund Experience',
       'Refund Issues and Customer Service', 'Declining Service Quality',
       'Very Positive Feedback',
       'Refund Issues and Misdelivered Products',
       'Worst Apps and Applications', 'Quick_and_Fast_Delivery',
       'Positive Feedback and Exclamations',
       'Worst Customer Service and Delivery Issues',
       'Melted Ice Cream Delivery Issues', 'Poor Customer Service',
       'Experience Quality', 'Delivery Partner Behavior and Service',
       'Positive Feedback and Experiences',
       'Customer Experience Issues with Blinkit Delivery Service',
       'Fast and Quality Deliv

In [39]:
merged['global_generalized_topic'].unique()

array(['Customer Service Issues', 'User Experience Issues',
       'Noise / Unclustered', 'Expired and Damaged Products',
       'Service Quality Issues', 'Delivery Speed and Reliability',
       'Location Services Issues', 'Delivery Timeliness Issues',
       'Delivery and Refund Issues', 'Positive Feedback',
       'App Quality Issues', 'Delivery Speed',
       'Positive Feedback and Exclamations', 'Product Condition Issues',
       'Delivery Service Issues',
       'Customer Experience Issues with Blinkit Delivery Service',
       'Delivery Speed and Quality', 'Operational Cost Issues',
       'Product Quality Issues', 'Outstanding Performance and Experience',
       'Fees and Charges Issues', 'Order and Pricing Issues',
       'Customer Feedback', 'Positive Feedback and Emotions',
       'Best and Worst Sellers in Books',
       'Challenges and Opportunities in Digital Marketing Jobs',
       'Legal and Compliance Issues',
       'Video Content Engagement and Passion'], dtype=objec